## STEP: 7 FEATURE ENGINEERING
| Feature                   | Meaning                                            |
| ------------------------- | -------------------------------------------------- |
| `total_order_value`       | Total payment value for an order                   |
| `delivery_days`           | Actual number of days taken to deliver             |
| `delivery_delay`          | Days delayed compared with estimated delivery      |
| `customer_order_count`    | Number of orders placed by the customer            |
| `customer_total_spending` | Total amount spent by the customer                 |
| `average_order_value`     | Customer's average spending per order              |
| `seller_revenue`          | Total revenue generated by the seller              |
| `seller_order_count`      | Number of orders handled by the seller             |
| `repeat_customer`         | 1 if customer has more than one order, otherwise 0 |

| # | Feature                        | Definition                                                |
| - | ------------------------------ | --------------------------------------------------------- |
| 1 | **Total Order Value**          | Sum of `price + freight_value` for all items in an order  |
| 2 | **Actual Delivery Days**       | Delivered customer date − order purchase date             |
| 3 | **Delivery Delay Days**        | Actual delivery date − estimated delivery date            |
| 4 | **Customer Order Count**       | Number of unique orders placed by the customer            |
| 5 | **Customer Total Spending**    | Sum of `price + freight_value` across all customer orders |
| 6 | **Average Spending per Order** | Customer total spending ÷ customer order count            |
| 7 | **Seller Revenue**             | Sum of `price` for products sold by the seller            |
| 8 | **Seller Order Count**         | Number of unique orders handled by the seller             |
| 9 | **Repeat Customer**            | `1` if customer has more than one order, otherwise `0`    |



In [4]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="MySQL@2026Strong",
    database="cart2"
)

cursor = conn.cursor()

print("Connected to cart2 successfully")

Connected to cart2 successfully


In [5]:
query = """
CREATE TABLE business_features AS

WITH
order_value AS (
    SELECT
        order_id,
        ROUND(SUM(payment_value), 2) AS total_order_value
    FROM order_payments
    GROUP BY order_id
),

customer_features AS (
    SELECT
        c.customer_id,
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS customer_order_count,
        ROUND(SUM(ov.total_order_value), 2) AS customer_total_spending,
        ROUND(AVG(ov.total_order_value), 2) AS average_order_value,
        CASE
            WHEN COUNT(DISTINCT o.order_id) > 1 THEN 1
            ELSE 0
        END AS repeat_customer
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_value ov
        ON o.order_id = ov.order_id
    GROUP BY
        c.customer_id,
        c.customer_unique_id
),

seller_features AS (
    SELECT
        seller_id,
        ROUND(SUM(price), 2) AS seller_revenue,
        COUNT(DISTINCT order_id) AS seller_order_count
    FROM order_items
    GROUP BY seller_id
)

SELECT
    o.order_id,
    o.customer_id,
    ov.total_order_value,
    o.actual_delivery_days AS delivery_days,
    o.delivery_delay_days AS delivery_delay,
    cf.customer_order_count,
    cf.customer_total_spending,
    cf.average_order_value,
    cf.repeat_customer,
    sf.seller_revenue,
    sf.seller_order_count

FROM orders o

LEFT JOIN order_value ov
    ON o.order_id = ov.order_id

LEFT JOIN customer_features cf
    ON o.customer_id = cf.customer_id

LEFT JOIN (
    SELECT
        order_id,
        seller_id
    FROM order_items
    GROUP BY order_id, seller_id
) oi
    ON o.order_id = oi.order_id

LEFT JOIN seller_features sf
    ON oi.seller_id = sf.seller_id;
"""

cursor.execute(query)
conn.commit()

print("business_features table created successfully")

business_features table created successfully


In [6]:
cursor.execute("""
DESCRIBE business_features;
""")

for row in cursor.fetchall():
    print(row)

('order_id', 'varchar(32)', 'NO', '', None, '')
('customer_id', 'varchar(32)', 'YES', '', None, '')
('total_order_value', 'double', 'YES', '', None, '')
('delivery_days', 'int', 'YES', '', None, '')
('delivery_delay', 'int', 'YES', '', None, '')
('customer_order_count', 'bigint', 'YES', '', '0', '')
('customer_total_spending', 'double', 'YES', '', None, '')
('average_order_value', 'double', 'YES', '', None, '')
('repeat_customer', 'int', 'YES', '', '0', '')
('seller_revenue', 'double', 'YES', '', None, '')
('seller_order_count', 'bigint', 'YES', '', '0', '')


In [7]:
cursor.execute("""
SELECT *
FROM business_features
LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)

('00010242fe8c5a6d1ba2dd792cb16214', '3ce436f183e68e07877b285a838db11a', 72.19, 8, -8, 1, 72.19, 72.19, 0, 12271.71, 138)
('00018f77f2f0320c557190d7a144bdd3', 'f6dd3ec061db4e3987629fe6b26e5cce', 259.83, 16, -2, 1, 259.83, 259.83, 0, 9178.51, 122)
('000229ec398224ef6ca0657da4fc703e', '6489ae5e4333f3693df5ad4372dab6d3', 216.87, 8, -13, 1, 216.87, 216.87, 0, 3280.0, 12)
('00024acbcdf0a6daa1e931b038114c75', 'd4eb9395c8c0431ee92fce09860c5a06', 25.78, 6, -5, 1, 25.78, 25.78, 0, 1054.82, 13)
('00042b26cf59d7ce69dfabb4e55b4fd9', '58dbd0b2d70206bf40e62cd34e84d795', 218.04, 25, -15, 1, 218.04, 218.04, 0, 3661.18, 28)
('00048cc3ae777c65dbb7d2a0634bc1ea', '816cbea969fe5b689b39cfc97a506742', 34.59, 7, -14, 1, 34.59, 34.59, 0, 1209.64, 23)
('00054e8431b9d7675808bcb819fb4a32', '32e2e6ab09e778d99bf2e0ecd4898718', 31.75, 8, -16, 1, 31.75, 31.75, 0, 9738.2, 212)
('000576fe39319847cbb9d288c5617fa6', '9ed5e522dd9dd85b4af4a077526d8117', 880.75, 5, -15, 1, 880.75, 880.75, 0, 810.0, 1)
('0005a1a1728c9d785b8e

In [8]:
cursor.execute("""
SELECT COUNT(*) AS total_rows
FROM business_features;
""")

print(cursor.fetchone())

(100785,)


In [9]:
cursor.execute("""
SELECT COUNT(DISTINCT order_id) AS unique_orders
FROM business_features;
""")

print(cursor.fetchone())

(99441,)


In [10]:
cursor.execute("""
SELECT COUNT(DISTINCT order_id) AS unique_orders
FROM business_features;
""")

print(cursor.fetchone())

(99441,)


In [ ]:
Verification completed

Table created
All required business features present
99,441 total rows
99,441 unique orders
No unwanted duplicate orders
Delivery delay preserves negative values for early deliveries
Customer and seller features successfully calculated

Feature Engineering: 
Created meaningful business features including order value, delivery performance, customer purchasing behavior, repeat-customer indicators, 
and seller performance metrics using SQL aggregations, CTEs, joins, and conditional logic.

In [11]:
import pandas as pd

query = """
SELECT *
FROM business_features
LIMIT 20;
"""

df_features = pd.read_sql(query, conn)

df_features

C:\Users\VIDYAPRIYA V\AppData\Local\Temp\ipykernel_19300\3496401297.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_features = pd.read_sql(query, conn)


,order_id,customer_id,total_order_value,delivery_days,delivery_delay,customer_order_count,customer_total_spending,average_order_value,repeat_customer,seller_revenue,seller_order_count
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,72.19,8,-8,1,72.19,72.19,0,12271.71,138
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,259.83,16,-2,1,259.83,259.83,0,9178.51,122
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,216.87,8,-13,1,216.87,216.87,0,3280.00,12
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,25.78,6,-5,1,25.78,25.78,0,1054.82,13
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,218.04,25,-15,1,218.04,218.04,0,3661.18,28
5,00048cc3ae777c65dbb7d2a0634bc1ea,816cbea969fe5b689b39cfc97a506742,34.59,7,-14,1,34.59,34.59,0,1209.64,23
6,00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,31.75,8,-16,1,31.75,31.75,0,9738.20,212
7,000576fe39319847cbb9d288c5617fa6,9ed5e522dd9dd85b4af4a077526d8117,880.75,5,-15,1,880.75,880.75,0,810.00,1
8,0005a1a1728c9d785b8e2b08b904576c,16150771dfd4776261284213b89c304e,157.60,10,1,1,157.60,157.60,0,25670.72,162
9,0005f50442cb953dcd1d21e1fb923495,351d3cb2cee3c7fd0af6616c82df21d3,65.39,2,-18,1,65.39,65.39,0,6110.58,85


In [12]:
df_features.shape

(20, 11)

In [16]:
## 1.Total payment value for an order
query = """
SELECT
    order_id,
    ROUND(SUM(payment_value), 2) AS total_order_value
FROM order_payments
GROUP BY order_id
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('00010242fe8c5a6d1ba2dd792cb16214', 72.19)
('00018f77f2f0320c557190d7a144bdd3', 259.83)
('000229ec398224ef6ca0657da4fc703e', 216.87)
('00024acbcdf0a6daa1e931b038114c75', 25.78)
('00042b26cf59d7ce69dfabb4e55b4fd9', 218.04)
('00048cc3ae777c65dbb7d2a0634bc1ea', 34.59)
('00054e8431b9d7675808bcb819fb4a32', 31.75)
('000576fe39319847cbb9d288c5617fa6', 880.75)
('0005a1a1728c9d785b8e2b08b904576c', 157.6)
('0005f50442cb953dcd1d21e1fb923495', 65.39)


In [17]:
import pandas as pd

df = pd.read_sql(query, conn)
df

C:\Users\VIDYAPRIYA V\AppData\Local\Temp\ipykernel_19300\1865705339.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,order_id,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,34.59
6,00054e8431b9d7675808bcb819fb4a32,31.75
7,000576fe39319847cbb9d288c5617fa6,880.75
8,0005a1a1728c9d785b8e2b08b904576c,157.60
9,0005f50442cb953dcd1d21e1fb923495,65.39


In [18]:
query = """
SELECT
    order_id,
    ROUND(SUM(price + freight_value), 2) AS total_order_value
FROM order_items
GROUP BY order_id
LIMIT 10;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('00010242fe8c5a6d1ba2dd792cb16214', 72.19)
('00018f77f2f0320c557190d7a144bdd3', 259.83)
('000229ec398224ef6ca0657da4fc703e', 216.87)
('00024acbcdf0a6daa1e931b038114c75', 25.78)
('00042b26cf59d7ce69dfabb4e55b4fd9', 218.04)
('00048cc3ae777c65dbb7d2a0634bc1ea', 34.59)
('00054e8431b9d7675808bcb819fb4a32', 31.75)
('000576fe39319847cbb9d288c5617fa6', 880.75)
('0005a1a1728c9d785b8e2b08b904576c', 157.6)
('0005f50442cb953dcd1d21e1fb923495', 65.39)


In [19]:
## 2. Actual number of days taken to deliver
query = """
SELECT
    order_id,
    DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) AS actual_delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('00010242fe8c5a6d1ba2dd792cb16214', 7)
('00018f77f2f0320c557190d7a144bdd3', 16)
('000229ec398224ef6ca0657da4fc703e', 8)
('00024acbcdf0a6daa1e931b038114c75', 6)
('00042b26cf59d7ce69dfabb4e55b4fd9', 25)
('00048cc3ae777c65dbb7d2a0634bc1ea', 7)
('00054e8431b9d7675808bcb819fb4a32', 8)
('000576fe39319847cbb9d288c5617fa6', 5)
('0005a1a1728c9d785b8e2b08b904576c', 10)
('0005f50442cb953dcd1d21e1fb923495', 2)
('00061f2a7bc09da83e415a52dc8a4af1', 5)
('00063b381e2406b52ad429470734ebd5', 11)
('0006ec9db01a64e59a68b2c340bf65a7', 7)
('0008288aa423d2a3f00fcb17cd7d8719', 13)
('0009792311464db532ff765bf7b182ae', 8)
('0009c9a17f916a706d71784483a5d643', 5)
('000aed2e25dbad2f9ddb70584c5a2ded', 7)
('000c3e6612759851cc3cbb4b83257986', 7)
('000e562887b1f2006d75e0be9558292e', 18)
('000e63d38ae8c00bbcb5a30573b99628', 4)
('000e906b789b55f64edcb1f84030f90d', 18)
('000f25f4d72195062c040b12dce9a18a', 15)
('001021efaa8636c29475e7734483457d', 9)
('0010b2e5201cc5f1ae7e9c6cc8f5bd00', 12)
('00119ff934e539cf26f92b9ef0cdf

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [20]:
query = """
UPDATE orders
SET actual_delivery_days = DATEDIFF(
    order_delivered_customer_date,
    order_purchase_timestamp
)
WHERE order_delivered_customer_date IS NOT NULL;
"""

cursor.execute(query)
conn.commit()

print("Actual delivery days calculated successfully")

Actual delivery days calculated successfully


In [21]:
## 3. Days delayed compared with estimated delivery
query = """
SELECT
    order_id,
    DATEDIFF(
        order_delivered_customer_date,
        order_estimated_delivery_date
    ) AS delivery_delay_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('00010242fe8c5a6d1ba2dd792cb16214', -9)
('00018f77f2f0320c557190d7a144bdd3', -3)
('000229ec398224ef6ca0657da4fc703e', -14)
('00024acbcdf0a6daa1e931b038114c75', -6)
('00042b26cf59d7ce69dfabb4e55b4fd9', -16)
('00048cc3ae777c65dbb7d2a0634bc1ea', -15)
('00054e8431b9d7675808bcb819fb4a32', -17)
('000576fe39319847cbb9d288c5617fa6', -16)
('0005a1a1728c9d785b8e2b08b904576c', 0)
('0005f50442cb953dcd1d21e1fb923495', -19)
('00061f2a7bc09da83e415a52dc8a4af1', -11)
('00063b381e2406b52ad429470734ebd5', 0)
('0006ec9db01a64e59a68b2c340bf65a7', -22)
('0008288aa423d2a3f00fcb17cd7d8719', -8)
('0009792311464db532ff765bf7b182ae', -6)
('0009c9a17f916a706d71784483a5d643', -9)
('000aed2e25dbad2f9ddb70584c5a2ded', -4)
('000c3e6612759851cc3cbb4b83257986', -13)
('000e562887b1f2006d75e0be9558292e', -7)
('000e63d38ae8c00bbcb5a30573b99628', -9)
('000e906b789b55f64edcb1f84030f90d', 2)
('000f25f4d72195062c040b12dce9a18a', -20)
('001021efaa8636c29475e7734483457d', -15)
('0010b2e5201cc5f1ae7e9c6cc8f5bd00', -4)
('00119f

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



More than 1 order → 1 (Repeat Customer)
1 order           → 0 (Not Repeat Customer)